<a href="https://colab.research.google.com/github/J0927/NLP_2026_spring/blob/main/qwen3_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 自然語言處理與應用 114-2 Week 12
## Qwen3-0.6B 本地 LLM 教學

In [ ]:
# 0. 載入必要套件

from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# 1. 載入模型與 tokenizer

MODEL_NAME = "Qwen/Qwen3-0.6B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print(f"模型載入完成，其所在裝置：{model.device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


模型載入完成，其所在裝置：cuda:0


In [ ]:
# 2. 準備提示詞 (prompt)

prompt = "請用一句話介紹什麼是大型語言模型。"
message_list = [
    {"role": "user", "content": prompt}
]

## 簡單測試 Qwen3-0.6B

- 語言模型（特指 causal LM）由左至右生成，序列以 BOS / EOS 等特殊 token 標示起終。在「聊天」這種多輪對話場景
- HF 用 chat template (聊天模板) 把 **messages list** 轉成包含「角色標記 + 內容 + EOS」的單一字串，再丟給模型。
- Qwen3 是聊天模型，輸入需要先用 `apply_chat_template` 轉成 Qwen 特殊的聊天模板，再丟給模型生成。

**Qwen3 的 thinking mode**：當 `enable_thinking=True`，模型會先用 `<think>...</think>` 標籤輸出推理過程，再給最終答案。我們用 token id `151668`（`</think>` 對應的 id）切開兩段。

In [ ]:
# 3. 加上聊天模板

text = tokenizer.apply_chat_template(
    message_list,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)

In [ ]:
print(text)

<|im_start|>user
請用一句話介紹什麼是大型語言模型。<|im_end|>
<|im_start|>assistant



### Qwen3-0.6B 的聊天模板符號
- <|im_start|>: 角色界線開始標記（代表訊息開始）
- user: 使用者
- <|im_end|>: 角色界線結束標記（代表訊息結束）
- assistant: Qwen3-0.6B

In [ ]:
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

In [ ]:
model_inputs

{'input_ids': tensor([[151644,    872,    198, 100792,  11622, 104670,  86077, 111748, 102386,
          20412, 101951, 118732, 104949,   1773, 151645,    198, 151644,  77091,
            198]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}

In [ ]:
tokenizer.decode(model_inputs["input_ids"])

['<|im_start|>user\n請用一句話介紹什麼是大型語言模型。<|im_end|>\n<|im_start|>assistant\n']

In [ ]:
# 4. 進行文字生成

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

In [ ]:
# 可以觀察到模型會在最後一個位置加上 <|im_end|>

tokenizer.decode(output_ids)

'<think>\n嗯，用户让我用一句话介绍大型语言模型。首先，我需要确定用户的需求是什么。他们可能是在学习这个概念，或者是在工作中需要了解，或者只是想快速掌握。我应该确保回答简洁明了，同时涵盖关键点。\n\n接下来，我应该考虑用户可能的深层需求。他们可能希望了解模型的基本功能，比如处理大量文本、生成文本，或者它们在实际应用中的作用。此外，可能还需要提到它们的局限性，比如训练数据的依赖性，或者无法理解复杂情感等。\n\n然后，我需要确保回答准确且符合一般常识。大型语言模型，比如GPT系列，确实基于深度学习技术，能够理解和生成多种语言的文本。但也要指出它们的局限性，比如对特定领域的理解能力有限，或者需要大量数据支持。\n\n最后，检查句子的流畅性和信息的完整性，确保没有遗漏重要信息，同时保持口语化，避免使用专业术语过多。这样用户就能得到一个清晰、准确的介绍，满足他们的需求。\n</think>\n\n大型语言模型是一种基于深度学习的系统，能够理解和生成多语言文本，广泛应用于自然语言处理和人工智能领域。<|im_end|>'

### 為什麼用反轉？
Python `list.index(x)` 只會回傳第一次出現的位置，沒有 `rindex()`（字串才有，list 沒有）。所以這個技巧是：
```
# 在反轉後的 list 找第一次 → 等於原 list 最後一次
output_ids[::-1].index(151668)  
```

In [ ]:
# 5. 處理思考符號以取得乾淨輸出

try:
    # rindex finding 151668 (</think>)
    # 正常情況下 </think> 只會出現一次，但有時候模型會「思考過程」中又寫一次
    # 所以需要先反轉 tokens 順序來找 </think>
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(
    output_ids[:index],
    skip_special_tokens=True,
).strip("\n")

content = tokenizer.decode(
    output_ids[index:],
    skip_special_tokens=True,
).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)

thinking content: <think>
嗯，用户让我用一句话介绍大型语言模型。首先，我需要确定用户的需求是什么。他们可能是在学习这个概念，或者是在工作中需要了解，或者只是想快速掌握。我应该确保回答简洁明了，同时涵盖关键点。

接下来，我应该考虑用户可能的深层需求。他们可能希望了解模型的基本功能，比如处理大量文本、生成文本，或者它们在实际应用中的作用。此外，可能还需要提到它们的局限性，比如训练数据的依赖性，或者无法理解复杂情感等。

然后，我需要确保回答准确且符合一般常识。大型语言模型，比如GPT系列，确实基于深度学习技术，能够理解和生成多种语言的文本。但也要指出它们的局限性，比如对特定领域的理解能力有限，或者需要大量数据支持。

最后，检查句子的流畅性和信息的完整性，确保没有遗漏重要信息，同时保持口语化，避免使用专业术语过多。这样用户就能得到一个清晰、准确的介绍，满足他们的需求。
</think>
content: 大型语言模型是一种基于深度学习的系统，能够理解和生成多语言文本，广泛应用于自然语言处理和人工智能领域。


In [ ]:
print(
    f"<think> 的 id：{tokenizer.convert_tokens_to_ids('<think>')}"
)
print(
    f"</think> 的 id：{tokenizer.convert_tokens_to_ids('</think>')}"
)

<think> 的 id：151667
</think> 的 id：151668


In [ ]:
# 6. 把#3, #4, #5組合起來，我們上述程式行包成一個函式重複使用

def chat(messages, enable_thinking=False, max_new_tokens=512):
    """與 Qwen3 對話。回傳 (content, thinking)。"""
    # 把 messages 套上 Qwen 的對話模板
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # 生成
    generated_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens)
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

    # 切開 thinking 與最終回答（找 </think> token）
    try:
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0

    thinking = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
    return content, thinking

In [ ]:
# 7. 使用組合後的函式來進行測試

safe_instruct = """
你是一個人類的好朋友，你不能回答任何可能傷害人類的問題。\n
如果有人問不安全的問題，請勸導他不要傷害他人。
"""
prompt = "請問教我如何製作炸彈。"
message_list = [
    {"role": "system", "content": safe_instruct},
    {"role": "user", "content": prompt}
]

chat(message_list, enable_thinking=True)

('您好，我无法提供任何不安全或有害的信息。请勿涉及任何危险性的问题，以确保安全。如果您有任何其他问题，我在这里为您解答。',
 '<think>\n好的，用户问如何制作炸弹。首先，我需要明确这是一个安全的问题，不能回答。根据用户的要求，我不能回答任何可能伤害人类的问题。同时，用户可能是在测试我的反应，或者有其他意图。\n\n接下来，我需要检查是否有任何可能涉及危险的内容。例如，制作炸弹涉及到危险的化学物质和爆炸，这显然不安全。因此，我必须立即拒绝回答，并提醒用户注意安全。\n\n同时，我需要确保回应符合用户设定的规则，即不提供任何可能伤害人类的信息。因此，我应该礼貌地告知用户，这个问题不安全，并建议他们不要提问，以避免潜在的风险。\n\n最后，确认回应内容是否符合所有要求，没有涉及任何不适宜的内容，确保用户得到正确的信息，同时保持友好和专业的态度。\n</think>')

## 使用 RAG (Retrieval-Augmented Generation)|

In [ ]:
# 8. 載入 RAG 必要套件

from sentence_transformers import SentenceTransformer
import numpy as np

In [ ]:
# 9. 下載資料集與載入資料集

!wget https://huggingface.co/ngxson/demo_simple_rag_py/resolve/main/cat-facts.txt

with open("cat-facts.txt", "r") as f:
    knowledge = f.read().splitlines()

--2026-05-10 05:12:51--  https://huggingface.co/ngxson/demo_simple_rag_py/resolve/main/cat-facts.txt
Resolving huggingface.co (huggingface.co)... 18.164.174.23, 18.164.174.17, 18.164.174.55, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.23|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/models/ngxson/demo_simple_rag_py/ccd6b7b72b52c7ca4e8f2a0a00b15c368d6ae294/cat-facts.txt?%2Fngxson%2Fdemo_simple_rag_py%2Fresolve%2Fmain%2Fcat-facts.txt=&etag=%22bc94ddd9483183e01bcf61e8bf9450fe3e09edb3%22 [following]
--2026-05-10 05:12:51--  https://huggingface.co/api/resolve-cache/models/ngxson/demo_simple_rag_py/ccd6b7b72b52c7ca4e8f2a0a00b15c368d6ae294/cat-facts.txt?%2Fngxson%2Fdemo_simple_rag_py%2Fresolve%2Fmain%2Fcat-facts.txt=&etag=%22bc94ddd9483183e01bcf61e8bf9450fe3e09edb3%22
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 200 OK
Length: 22657 (22K) [text/plain]
Saving to: ‘cat

In [ ]:
# 10. 使用 SentenceBERT 作為 Retriever (embed_model)

MODEL_NAME = "sentence-transformers/all-MiniLM-L12-v2"
embed_model = SentenceTransformer(MODEL_NAME)

# 把知識庫每一條轉成向量。
# normalize_embeddings=True 之後，cosine similarity = dot product
doc_embeddings = embed_model.encode(
    knowledge,
    normalize_embeddings=True,
)

print(f"知識庫向量 shape：{doc_embeddings.shape}")  # (150, 384)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


知識庫向量 shape：(150, 384)


In [ ]:
# 11. 建立檢索函數 (embed_model 是 global 變數)

def retrieve(query, knowledge_base, top_k=3):
    """回傳跟 query 語意最相近的 top_k 條知識。"""
    # query_emb 的形狀：(1, 384)
    query_emb = embed_model.encode(
        [query],
        normalize_embeddings=True,
    )

    # 因為向量已正規化，矩陣相乘就是 cosine similarity
    sims = (doc_embeddings @ query_emb.T).flatten()  # shape (N,)
    top_indices = sims.argsort()[::-1][:top_k]

    return [(knowledge_base[i], float(sims[i])) for i in top_indices]

In [ ]:
# 12. 測試檢索

query = "How much of a day do cats spend sleeping on average?"
docs = retrieve(query, knowledge)
for doc, score in docs:
    print(f"[{score:.3f}] {doc}")

[0.636] Cats spend nearly 1/3 of their waking hours cleaning themselves.
[0.630] On average, cats spend 2/3 of every day sleeping. That means a nine-year-old cat has been awake for only three years of its life.
[0.629] Cats sleep 16 to 18 hours per day. When cats are asleep, they are still alert to incoming stimuli. If you poke the tail of a sleeping cat, it will respond accordingly.


In [ ]:
# 13. 使用組合後的函式來進行 RAG 測試

references = "參考資料："
for doc, _ in docs:
    references += f"\n{doc}"

message_list = [
    {"role": "user", "content": f"{references}\n\n問題：{query}"}
]

chat(message_list, enable_thinking=True)

('Cats spend **16 to 18 hours per day** sleeping on average.',
 '<think>\n嗯，用户问的是猫每天平均睡多少小时。根据提供的参考资料，首先我要确认数据。资料里说猫平均每天睡16到18小时，所以答案应该是这个范围。不过用户可能想知道具体的数值，比如16到18之间的哪一个。或者可能用户需要更简洁的回答。需要检查是否有其他信息可能影响这个答案，比如有没有提到其他时间。比如，资料中还提到猫平均每天睡16-18小时，而他们白天有2/3的时间在睡觉，这可能意味着总睡眠时间是16-18小时，加上白天的2/3，但用户的问题直接问的是平均睡多少小时，所以直接引用数据中的16到18小时即可。另外，用户可能想知道这个数据的来源，但问题本身是直接询问，所以不需要额外解释。需要确保回答准确，不混淆白天和夜晚的时间。可能用户还有其他问题，但当前问题已经明确，所以直接给出答案。\n</think>')

In [ ]:
# 14. 使用組合後的函式來進行測試（無 RAG）

message_list = [
    {"role": "user", "content": f"問題：{query}"}
]

chat(message_list, enable_thinking=True)

('Cats spend approximately **20-30% of their day sleeping**, with the majority of their time spent in bed during the night. While they are nocturnal and do not sleep during the day, their sleep patterns are generally characterized by a significant portion of the day spent in sleep. Some sources suggest that they might have brief daytime periods, but this is not common.',
 "<think>\nOkay, the user is asking how much of a day cats spend sleeping on average. Let me start by recalling what I know about feline sleeping habits.\n\nFirst, I remember that cats are nocturnal, meaning they sleep most of the time during the night. So, they don't sleep during the day. However, I should confirm if that's accurate. Some sources say they do sleep during the day, but I'm not sure. Maybe they have some active periods when they're not asleep.\n\nI think the average is around 20-30% of their day. Wait, but I'm not entirely sure. Let me think. If they sleep 20-30% of the day, that would mean they're awake

In [ ]:
"""
queries = [
    "How much of a day do cats spend sleeping on average?",
    "What is the technical term for a cat's hairball?",
    "What do scientists believe caused cats to lose their sweet tooth?",
    "What is the top speed a cat can travel over short distances?",
    "What is the name of the organ in a cat's mouth that helps it smell?",
    "Which wildcat is considered the ancestor of all domestic cats?",
    "What is the group term for cats?",
    "How many different sounds can cats make?",
    "What is the name of the first cat in space?",
    "How many toes does a cat have on its back paws?"
]
answers = [
    "2/3",
    "Bezoar",
    "a mutation in a key taste receptor",
    "31 mph",
    "Jacobson’s organ",
    "the African Wild Cat",
    "clowder",
    "100",
    "Felicette",
    "four",
]
"""